# Customer Churn Prediction - PySpark MLlib

**End-to-end reproducible workflow** | IBM Telco Customer Churn | Classification

This notebook complements the scikit-learn implementation. It uses a **stratified train / validation / test split**, fits preprocessing on training data only, selects a model and classification threshold on validation data, and evaluates the final result on an untouched test set. **Metrics must be computed by running the notebook**; no results are assumed.

> For an apples-to-apples comparison, both notebooks must use the same rows, features, split, model settings and selection protocol. This notebook uses a separate validation set, so its scores are **not directly comparable** to the previously reported scikit-learn test scores.

## 0 - Setup
Run in Google Colab. PySpark installation may take a few minutes.

In [2]:
%pip -q install pyspark scikit-learn pandas matplotlib

In [3]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, f1_score, classification_report, confusion_matrix, roc_auc_score, average_precision_score
from pyspark.sql import SparkSession, functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator

SEED = 42
spark = SparkSession.builder.appName("CustomerChurnMLlib").getOrCreate()
print("Spark version:", spark.version)

Spark version: 4.0.4


## 1 - Load and validate source data

In [4]:
DATA_URL = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
local_path = Path("data/Telco-Customer-Churn.csv")
pdf = pd.read_csv(local_path if local_path.exists() else DATA_URL)
assert {"customerID", "TotalCharges", "tenure", "Churn"}.issubset(pdf.columns)
assert pdf["customerID"].is_unique
assert pdf["Churn"].isin(["Yes", "No"]).all()
print("Rows and columns:", pdf.shape)
display(pdf.head(3))

Rows and columns: (7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes


## 2 - Documented data cleaning
Blank TotalCharges values are assumed to represent zero accumulated charges **only when tenure is zero**. We retain an indicator of original missingness. This is a domain assumption, not verified billing information.

In [5]:
pdf["TotalCharges"] = pd.to_numeric(pdf["TotalCharges"], errors="coerce")
pdf["TotalCharges_missing"] = pdf["TotalCharges"].isna().astype(int)
missing = pdf["TotalCharges"].isna()
assert pdf.loc[missing, "tenure"].eq(0).all(), "Revisit missing-value policy"
pdf.loc[missing, "TotalCharges"] = 0.0
pdf["label"] = (pdf["Churn"] == "Yes").astype(float)
print("Original missing TotalCharges:", missing.sum())
print("Remaining nulls:", pdf.isna().sum().sum())

Original missing TotalCharges: 11
Remaining nulls: 0


## 3 - Stratified splits
We use **64% train / 16% validation / 20% test** with the same random seed. Validation is used for model and threshold selection; test is used once for final reporting.

In [6]:
train_val, test_pd = train_test_split(pdf, test_size=0.20, stratify=pdf["label"], random_state=SEED)
train_pd, val_pd = train_test_split(train_val, test_size=0.20, stratify=train_val["label"], random_state=SEED)
print({"train": len(train_pd), "validation": len(val_pd), "test": len(test_pd)})
print({"train_rate": train_pd.label.mean(), "val_rate": val_pd.label.mean(), "test_rate": test_pd.label.mean()})
train = spark.createDataFrame(train_pd)
val = spark.createDataFrame(val_pd)
test = spark.createDataFrame(test_pd)

{'train': 4507, 'validation': 1127, 'test': 1409}
{'train_rate': np.float64(0.2653649877967606), 'val_rate': np.float64(0.2653061224489796), 'test_rate': np.float64(0.2654364797728886)}


## 4 - Feature engineering pipeline
`customerID`, `Churn` and `label` are excluded from model features. We fit categorical encoders on training data only. Numerical features are assembled and standardized for Logistic Regression; the same transformed feature vector is used for Random Forest.

In [7]:
numeric_cols = ["SeniorCitizen", "tenure", "MonthlyCharges", "TotalCharges", "TotalCharges_missing"]
categorical_cols = [c for c in pdf.columns if c not in numeric_cols + ["customerID", "Churn", "label"]]
indexers = [StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep") for c in categorical_cols]
encoder = OneHotEncoder(inputCols=[f"{c}_idx" for c in categorical_cols], outputCols=[f"{c}_ohe" for c in categorical_cols], handleInvalid="keep")
num_assembler = VectorAssembler(inputCols=numeric_cols, outputCol="numeric_features", handleInvalid="error")
scaler = StandardScaler(inputCol="numeric_features", outputCol="scaled_numeric", withStd=True, withMean=False)
feature_assembler = VectorAssembler(inputCols=["scaled_numeric"] + [f"{c}_ohe" for c in categorical_cols], outputCol="features")
preprocess = indexers + [encoder, num_assembler, scaler, feature_assembler]
print("Numerical:", numeric_cols)
print("Categorical:", categorical_cols)

Numerical: ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges', 'TotalCharges_missing']
Categorical: ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']


## 5 - Train candidate models

In [8]:
candidates = {
    "Logistic Regression": LogisticRegression(featuresCol="features", labelCol="label", maxIter=100, regParam=0.01),
    "Random Forest": RandomForestClassifier(featuresCol="features", labelCol="label", numTrees=100, maxDepth=7, seed=SEED)
}
evaluator = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderROC")
trained = {}
rows = []
for name, classifier in candidates.items():
    fitted = Pipeline(stages=preprocess + [classifier]).fit(train)
    pred_val = fitted.transform(val)
    auc = evaluator.evaluate(pred_val)
    rows.append({"model": name, "validation_roc_auc": auc})
    trained[name] = fitted
results = pd.DataFrame(rows).sort_values("validation_roc_auc", ascending=False)
display(results)
best_name = results.iloc[0]["model"]
model = trained[best_name]
print("Selected on validation:", best_name)

,model,validation_roc_auc
1,Random Forest,0.847689
0,Logistic Regression,0.846259


Selected on validation: Random Forest


## 6 - Tune the decision threshold on validation only
For a churn-retention use case, we can choose the threshold that maximizes F1 on validation. Business cost-sensitive thresholding would be preferable if outreach costs and missed churn costs were known.

In [9]:
def predictions_to_pandas(spark_df):
    return spark_df.select("label", "probability").toPandas().assign(
        score=lambda d: d["probability"].map(lambda v: float(v[1]))
    )

val_scores = predictions_to_pandas(model.transform(val))
threshold_rows = []
for threshold in np.arange(0.20, 0.81, 0.05):
    predicted = (val_scores["score"] >= threshold).astype(int)
    threshold_rows.append({
        "threshold": round(float(threshold), 2),
        "precision": precision_score(val_scores["label"], predicted, zero_division=0),
        "recall": recall_score(val_scores["label"], predicted, zero_division=0),
        "f1": f1_score(val_scores["label"], predicted, zero_division=0)
    })
threshold_table = pd.DataFrame(threshold_rows).sort_values(["f1", "threshold"], ascending=[False, True])
display(threshold_table.round(3))
selected_threshold = float(threshold_table.iloc[0]["threshold"])
print("Threshold chosen on validation:", selected_threshold)

,threshold,precision,recall,f1
4,0.40,0.637,0.645,0.641
3,0.35,0.581,0.709,0.639
2,0.30,0.543,0.763,0.634
5,0.45,0.700,0.555,0.619
1,0.25,0.497,0.799,0.613
0,0.20,0.461,0.860,0.600
6,0.50,0.699,0.428,0.531
7,0.55,0.725,0.334,0.458
8,0.60,0.781,0.251,0.380
9,0.65,0.810,0.171,0.282


Threshold chosen on validation: 0.4


## 7 - Final test evaluation (do not tune after viewing these results)

In [10]:
test_scores = predictions_to_pandas(model.transform(test))
y_true = test_scores["label"].astype(int)
y_score = test_scores["score"]
y_hat = (y_score >= selected_threshold).astype(int)
print("Selected model:", best_name)
print("Selected threshold:", selected_threshold)
print("Test ROC-AUC:", round(roc_auc_score(y_true, y_score), 4))
print("Test Average Precision:", round(average_precision_score(y_true, y_score), 4))
print("Test confusion matrix:\n", confusion_matrix(y_true, y_hat))
print(classification_report(y_true, y_hat, target_names=["No churn", "Churn"], zero_division=0))

Selected model: Random Forest
Selected threshold: 0.4
Test ROC-AUC: 0.8386
Test Average Precision: 0.6508
Test confusion matrix:
 [[870 165]
 [137 237]]
              precision    recall  f1-score   support

    No churn       0.86      0.84      0.85      1035
       Churn       0.59      0.63      0.61       374

    accuracy                           0.79      1409
   macro avg       0.73      0.74      0.73      1409
weighted avg       0.79      0.79      0.79      1409



## 8 - Save model and document results
Saving a Spark PipelineModel writes a directory, not a single `.joblib` file. Record the dataset source, seed, versions, model and threshold in the README. Avoid committing large model artifacts by default.

In [11]:
from pathlib import Path
output = "models/pyspark_customer_churn_pipeline"
Path("models").mkdir(exist_ok=True)
model.write().overwrite().save(output)
print("Saved Spark pipeline to:", output)
print("Spark version:", spark.version)
print("Selected threshold:", selected_threshold)

Saved Spark pipeline to: models/pyspark_customer_churn_pipeline
Spark version: 4.0.4
Selected threshold: 0.4


## Interpretation and next steps

Random Forest achieved the higher validation ROC-AUC (**0.8477**, versus **0.8463** for Logistic Regression) and was selected for the final evaluation. A threshold of **0.40** maximized F1 among the thresholds evaluated on the validation set (validation precision **0.637**, recall **0.645**, F1 **0.641**).

On the untouched test set, Random Forest achieved ROC-AUC **0.8386**, average precision **0.6508**, and accuracy **0.79**. For churners, precision was **0.59**, recall **0.63**, and F1-score **0.61**. The confusion matrix was **TN=870, FP=165, FN=137, TP=237**.

The model identifies a useful share of churners, although 137 of 374 churners in the test set were missed. A practical retention team would need to weigh missed customers against the cost of contacting false positives.

The scikit-learn and PySpark scores are not a controlled framework comparison because their split and threshold-selection protocols differ. A future experiment should align the same train, validation and test rows, feature handling and decision policy. Results describe predictive associations, not causes of churn.